# Spark SQL Lab: инкрементальная загрузка хранилища данных

В этом notebook реализована формальная и воспроизводимая логика:
- поиска партиций источников;
- безопасного создания path-based хранилища;
- инкрементального обновления витрин без повторного чтения уже обработанных партиций;
- технического аудита загрузок и контроля неизменности источников.

Все основные операции оформлены в виде функций. Сразу после каждой функции идёт отдельная ячейка с примером вызова и ожидаемым типом результата.

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import re
import shutil
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from typing import Any

PARTITION_NAME_PATTERN = re.compile(r"^(?P<key>[A-Za-z0-9_]+)='(?P<value>.*)'$")
FUTURE_DTTM = dt.datetime(9999, 12, 31, 0, 0, 0)

SOURCE_REGISTRY = {
    'client_cards_daily': {
        'source_id': 1,
        'source_description': 'Daily client card source.',
        'update_frequency': 'daily',
        'target_table': 'client_daily_attrs_scd2',
        'partition_key': 'row_actual_to',
        'columns': (
            'srv_mb_nflag',
            'cc_stoplist',
            'lne_tot_debt_int_ovrd_rub_amt',
            'lne_tot_debt_ovrd_rub_amt',
        ),
    },
    'credit_cards_info': {
        'source_id': 2,
        'source_description': 'Monthly credit card source.',
        'update_frequency': 'monthly',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'columns': (
            'client_income_amt',
            'oi_total_amt',
            'act_pl_os_rub_amt',
            'payroll_client_nflag',
            'inf_payroll_rub_amt',
            'legal_entity_amt',
            'inc_avg_risk_rub_amt',
            'otf_loan_rub_amt',
            'otf_fee_rub_amt',
            'inf_transfer_rub_amt',
            'cc_ever_nflag',
        ),
    },
    'deb_cards_info': {
        'source_id': 3,
        'source_description': 'Monthly debit card source.',
        'update_frequency': 'monthly',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'columns': (
            'onl_bank_active_1m_nfalg',
            'auto_pay_active_qty',
            'cl_income_1m_amt',
            'dep_acc_1st_open_dt',
            'wdr_cash_6m_amt',
            'cash_op_6m_amt',
            'cash_3m_qty',
            'lst_balance_amt',
            'card_active_1m_nflag',
        ),
    },
}

ATTRIBUTE_ID_BY_NAME = {
    attribute_name: attribute_id
    for attribute_id, attribute_name in enumerate(
        [
            column_name
            for source_meta in SOURCE_REGISTRY.values()
            for column_name in source_meta['columns']
        ],
        start=1,
    )
}

WAREHOUSE_TABLE_SPECS = {
    'dim_sources': {
        'columns': [
            ('source_id', 'INT'),
            ('source_name', 'STRING'),
            ('source_description', 'STRING'),
            ('update_frequency', 'STRING'),
            ('row_create_dtime', 'TIMESTAMP'),
            ('row_update_dtime', 'TIMESTAMP'),
            ('valid_from', 'TIMESTAMP'),
            ('valid_to', 'TIMESTAMP'),
        ],
        'partitioned_by': [],
    },
    'dim_attributes': {
        'columns': [
            ('attribute_id', 'INT'),
            ('attribute_name', 'STRING'),
            ('attribute_description', 'STRING'),
            ('data_type', 'STRING'),
            ('source_id', 'INT'),
            ('update_frequency', 'STRING'),
            ('row_create_dtime', 'TIMESTAMP'),
            ('row_update_dtime', 'TIMESTAMP'),
        ],
        'partitioned_by': [],
    },
    'load_log': {
        'columns': [
            ('load_id', 'BIGINT'),
            ('source_id', 'INT'),
            ('source_name', 'STRING'),
            ('source_partition_key', 'STRING'),
            ('source_partition_value', 'STRING'),
            ('target_table', 'STRING'),
            ('load_start_dtime', 'TIMESTAMP'),
            ('load_end_dtime', 'TIMESTAMP'),
            ('load_status', 'STRING'),
            ('rows_loaded', 'BIGINT'),
            ('error_message', 'STRING'),
        ],
        'partitioned_by': [],
    },
    'tech_source_partitions': {
        'columns': [
            ('source_id', 'INT'),
            ('source_name', 'STRING'),
            ('target_table', 'STRING'),
            ('partition_key', 'STRING'),
            ('partition_value', 'STRING'),
            ('partition_path', 'STRING'),
            ('manifest_fingerprint', 'STRING'),
            ('last_processed_status', 'STRING'),
            ('first_load_id', 'BIGINT'),
            ('last_load_id', 'BIGINT'),
            ('row_create_dtime', 'TIMESTAMP'),
            ('row_update_dtime', 'TIMESTAMP'),
        ],
        'partitioned_by': [],
    },
    'client_monthly_attrs_scd1': {
        'columns': [
            ('client_id', 'STRING'),
            ('attribute_id', 'INT'),
            ('attribute_value', 'STRING'),
            ('source_id', 'INT'),
            ('row_update_dtime', 'TIMESTAMP'),
            ('row_loading_id', 'BIGINT'),
            ('row_hash_val', 'STRING'),
            ('report_dt', 'STRING'),
        ],
        'partitioned_by': ['report_dt'],
    },
    'client_daily_attrs_scd2': {
        'columns': [
            ('client_id', 'STRING'),
            ('attribute_id', 'INT'),
            ('attribute_value', 'STRING'),
            ('row_actual_from', 'STRING'),
            ('source_id', 'INT'),
            ('row_update_dtime', 'TIMESTAMP'),
            ('row_loading_id', 'BIGINT'),
            ('row_hash_val', 'STRING'),
            ('row_actual_to', 'STRING'),
        ],
        'partitioned_by': ['row_actual_to'],
    },
}

## 1. Bootstrap системного Spark

Notebook должен одинаково работать из локального окружения `uv`, если Apache Spark уже установлен в системе.

In [ ]:
def bootstrap_system_spark_python(explicit_spark_home: str | Path | None = None) -> dict[str, str]:
    """
    Подготавливает Python-путь так, чтобы текущий интерпретатор мог импортировать системный `pyspark`.

    Функция предназначена для сценария, в котором Apache Spark уже установлен на машине отдельно
    от Python-окружения notebook. В таком режиме `uv` управляет зависимостями тестов и служебных
    библиотек, а сам `pyspark` подхватывается напрямую из системного Spark-дистрибутива.

    Порядок работы функции:
    1. Ищет каталог Spark в явном аргументе, затем в `SPARK_HOME`, затем в типовых путях macOS Homebrew.
    2. Проверяет наличие `pyspark.zip` и `py4j-*.zip`.
    3. Добавляет найденные пути в `sys.path`, не дублируя записи.
    4. Возвращает словарь с фактической конфигурацией bootstrap-этапа.

    Параметры:
        explicit_spark_home:
            Необязательный путь к каталогу `SPARK_HOME`. Если значение передано, оно имеет приоритет
            над переменной окружения и типовыми путями поиска.

    Возвращаемое значение:
        Словарь с ключами `spark_home`, `python_dir`, `pyspark_zip`, `py4j_zip`.

    Исключения:
        RuntimeError: если не удалось обнаружить корректный системный Spark.

    Пример входа:
        bootstrap_system_spark_python('/opt/homebrew/opt/apache-spark/libexec')

    Пример выхода:
        {
            'spark_home': '/opt/homebrew/opt/apache-spark/libexec',
            'python_dir': '/opt/homebrew/opt/apache-spark/libexec/python',
            'pyspark_zip': '/opt/homebrew/opt/apache-spark/libexec/python/lib/pyspark.zip',
            'py4j_zip': '/opt/homebrew/opt/apache-spark/libexec/python/lib/py4j-0.10.9.9-src.zip'
        }
    """
    candidate_paths: list[Path] = []
    if explicit_spark_home is not None:
        candidate_paths.append(Path(explicit_spark_home))
    spark_home_env = os.environ.get('SPARK_HOME')
    if spark_home_env:
        candidate_paths.append(Path(spark_home_env))
    candidate_paths.append(Path('/opt/homebrew/opt/apache-spark/libexec'))
    candidate_paths.extend(sorted(Path('/opt/homebrew/Cellar/apache-spark').glob('*/libexec'), reverse=True))

    for spark_home in candidate_paths:
        python_dir = spark_home / 'python'
        lib_dir = python_dir / 'lib'
        pyspark_zip = lib_dir / 'pyspark.zip'
        py4j_candidates = sorted(lib_dir.glob('py4j-*.zip'))
        if not pyspark_zip.exists() or not py4j_candidates:
            continue

        py4j_zip = py4j_candidates[0]
        os.environ['SPARK_HOME'] = str(spark_home)
        os.environ.setdefault(
            'JAVA_HOME',
            '/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home',
        )
        os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')
        os.environ.setdefault('PYSPARK_PYTHON', sys.executable)

        for entry in (str(python_dir), str(pyspark_zip), str(py4j_zip)):
            if entry not in sys.path:
                sys.path.insert(0, entry)

        return {
            'spark_home': str(spark_home),
            'python_dir': str(python_dir),
            'pyspark_zip': str(pyspark_zip),
            'py4j_zip': str(py4j_zip),
        }

    raise RuntimeError(
        'Системный Apache Spark не найден. Необходимо установить Spark или передать explicit_spark_home.'
    )

In [ ]:
bootstrap_summary = bootstrap_system_spark_python()
print(json.dumps(bootstrap_summary, ensure_ascii=False, indent=2))

In [ ]:
def create_spark_session(
    app_name: str = 'dbscoring-spark-sql-lab',
    warehouse_dir: str | Path | None = None,
    shuffle_partitions: int = 8,
):
    """
    Создаёт и возвращает локальную `SparkSession` с параметрами, удобными для SQL-centric ETL.

    Функция сознательно не прячет конфигурацию внутри глобального состояния. Она делает запуск
    воспроизводимым: bootstrap системного Spark выполняется явно, сетевой интерфейс привязывается
    к `127.0.0.1`, пользовательский warehouse-каталог может быть передан извне, а число партиций
    shuffle-операций контролируется параметром.

    Параметры:
        app_name:
            Логическое имя Spark-приложения. Оно попадает в логи и упрощает диагностику.
        warehouse_dir:
            Необязательный путь к каталогу Spark warehouse. Полезно для тестов и изолированных прогонов.
        shuffle_partitions:
            Количество партиций для shuffle-операций локального запуска.

    Возвращаемое значение:
        Инициализированный объект `SparkSession`.

    Пример входа:
        create_spark_session(app_name='dbscoring-tests', warehouse_dir='/tmp/spark-warehouse', shuffle_partitions=4)

    Пример выхода:
        <pyspark.sql.session.SparkSession object at 0x...>
    """
    bootstrap_system_spark_python()
    from pyspark.sql import SparkSession

    builder = (
        SparkSession.builder.master('local[2]')
        .appName(app_name)
        .config('spark.ui.enabled', 'false')
        .config('spark.sql.session.timeZone', 'UTC')
        .config('spark.driver.host', '127.0.0.1')
        .config('spark.driver.bindAddress', '127.0.0.1')
        .config('spark.sql.shuffle.partitions', str(shuffle_partitions))
        .config('spark.default.parallelism', str(shuffle_partitions))
        .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
        .config('spark.sql.execution.arrow.pyspark.enabled', 'false')
        .config('spark.driver.extraJavaOptions', '-Djava.security.manager=allow')
        .config('spark.executor.extraJavaOptions', '-Djava.security.manager=allow')
    )
    if warehouse_dir is not None:
        builder = builder.config('spark.sql.warehouse.dir', str(Path(warehouse_dir).resolve()))

    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel('WARN')
    return spark

In [ ]:
EXAMPLE_RUNTIME_DIR = TemporaryDirectory()
EXAMPLE_RUNTIME_ROOT = Path(EXAMPLE_RUNTIME_DIR.name)
example_spark = create_spark_session(
    app_name='dbscoring-function-examples',
    warehouse_dir=EXAMPLE_RUNTIME_ROOT / 'spark_metastore',
    shuffle_partitions=2,
)
print(example_spark.version)

## 2. Валидация режима и путей выполнения

В production-режиме notebook не должен удалять артефакты. В debug-режиме допускается полная очистка только специально выделенного debug-root каталога.

In [ ]:
def validate_run_mode(run_mode: str) -> str:
    """
    Проверяет допустимость режима выполнения ETL-процесса.

    Допустимы только два режима:
    - `production`: безопасный режим без удаления существующих артефактов;
    - `debug`: режим разработки и демонстрации, в котором разрешена полная очистка только debug-root.

    Параметры:
        run_mode:
            Строковое имя режима выполнения.

    Возвращаемое значение:
        Нормализованное значение режима (`production` или `debug`).

    Исключения:
        ValueError: если передан неизвестный режим.

    Пример входа:
        validate_run_mode('debug')

    Пример выхода:
        'debug'
    """
    normalized_mode = run_mode.strip().lower()
    if normalized_mode not in {'production', 'debug'}:
        raise ValueError("Допустимы только режимы 'production' и 'debug'.")
    return normalized_mode

In [ ]:
print(validate_run_mode('DEBUG'))

In [ ]:
def resolve_runtime_paths(
    sources_root: str | Path | None = None,
    warehouse_root: str | Path | None = None,
    run_mode: str = 'production',
    debug_root: str | Path | None = None,
) -> dict[str, str]:
    """
    Разрешает физические пути источников и каталога хранилища для текущего запуска.

    Функция ищет каталог источников в следующем порядке:
    1. явный аргумент `sources_root`;
    2. переменная окружения `DBSCORING_SOURCES_ROOT`;
    3. типовые каталоги относительно текущей рабочей директории.

    Аналогично определяется production-root хранилища. Для debug-режима используется отдельный
    каталог, чтобы исключить случайное разрушение production-артефактов. Сама функция пути не
    создаёт и ничего не удаляет: она только формирует и валидирует контракт запуска.

    Параметры:
        sources_root:
            Необязательный явный путь к директории `sources`.
        warehouse_root:
            Необязательный production-root каталога хранилища.
        run_mode:
            Один из режимов `production` или `debug`.
        debug_root:
            Необязательный отдельный root для debug-прогонов.

    Возвращаемое значение:
        Словарь с ключами `sources_root`, `production_warehouse_root`, `debug_warehouse_root`,
        `active_warehouse_root` и `run_mode`.

    Пример входа:
        resolve_runtime_paths('/tmp/sources', '/tmp/warehouse_prod', 'debug', '/tmp/warehouse_debug')

    Пример выхода:
        {
            'sources_root': '/tmp/sources',
            'production_warehouse_root': '/tmp/warehouse_prod',
            'debug_warehouse_root': '/tmp/warehouse_debug',
            'active_warehouse_root': '/tmp/warehouse_debug',
            'run_mode': 'debug'
        }
    """
    normalized_mode = validate_run_mode(run_mode)

    source_candidates: list[Path] = []
    if sources_root is not None:
        source_candidates.append(Path(sources_root))
    if os.environ.get('DBSCORING_SOURCES_ROOT'):
        source_candidates.append(Path(os.environ['DBSCORING_SOURCES_ROOT']))
    for base_dir in (Path.cwd(), *Path.cwd().parents):
        source_candidates.extend(
            [
                base_dir / 'source' / 'sources',
                base_dir / 'data' / 'sources',
                base_dir / 'sources',
            ]
        )

    resolved_sources_root = None
    for candidate in source_candidates:
        if candidate.exists():
            resolved_sources_root = candidate.resolve()
            break
    if resolved_sources_root is None:
        raise FileNotFoundError('Не удалось определить каталог с исходными данными sources.')

    production_root = Path(
        warehouse_root
        or os.environ.get('DBSCORING_WAREHOUSE_ROOT')
        or (Path.cwd() / 'dbscoring' / 'warehouse_spark_sql')
    ).resolve()
    debug_root_path = Path(
        debug_root
        or os.environ.get('DBSCORING_DEBUG_ROOT')
        or production_root.parent / f'{production_root.name}_debug'
    ).resolve()
    if production_root == debug_root_path:
        raise ValueError('Production-root и debug-root не должны совпадать.')

    active_root = debug_root_path if normalized_mode == 'debug' else production_root
    return {
        'sources_root': str(resolved_sources_root),
        'production_warehouse_root': str(production_root),
        'debug_warehouse_root': str(debug_root_path),
        'active_warehouse_root': str(active_root),
        'run_mode': normalized_mode,
    }

In [ ]:
EXAMPLE_SOURCES_ROOT = EXAMPLE_RUNTIME_ROOT / 'sources'
for source_name, source_meta in SOURCE_REGISTRY.items():
    partition_dir = EXAMPLE_SOURCES_ROOT / source_name / f"{source_meta['partition_key']}='2024-01-01'"
    partition_dir.mkdir(parents=True, exist_ok=True)
example_paths = resolve_runtime_paths(
    sources_root=EXAMPLE_SOURCES_ROOT,
    warehouse_root=EXAMPLE_RUNTIME_ROOT / 'warehouse_prod',
    run_mode='debug',
    debug_root=EXAMPLE_RUNTIME_ROOT / 'warehouse_debug',
)
print(json.dumps(example_paths, ensure_ascii=False, indent=2))

## 3. Поиск и контроль партиций источников

In [ ]:
def parse_partition_directory_name(partition_dir_name: str) -> tuple[str, str]:
    """
    Разбирает имя директории партиции вида `key='value'` на имя ключа и значение.

    Функция строго валидирует формат, потому что ошибка в имени каталога должна выявляться до того,
    как начнётся чтение тяжёлых parquet-файлов. Это особенно важно при petabyte-scale сценариях,
    где позднее обнаружение ошибки приводит к дорогим холостым прогонкам.

    Параметры:
        partition_dir_name:
            Имя директории партиции, например `report_dt='2024-01-31'`.

    Возвращаемое значение:
        Кортеж `(partition_key, partition_value)`.

    Исключения:
        ValueError: если имя каталога не соответствует требуемому шаблону.

    Пример входа:
        parse_partition_directory_name("report_dt='2024-01-31'")

    Пример выхода:
        ('report_dt', '2024-01-31')
    """
    match = PARTITION_NAME_PATTERN.match(partition_dir_name)
    if match is None:
        raise ValueError(
            f"Имя каталога партиции '{partition_dir_name}' не соответствует шаблону key='value'."
        )
    return match.group('key'), match.group('value')

In [ ]:
print(parse_partition_directory_name("report_dt='2024-01-31'"))

In [ ]:
def discover_source_partitions(
    sources_root: str | Path,
    source_registry: dict[str, dict[str, Any]] | None = None,
) -> list[dict[str, Any]]:
    """
    Находит все доступные партиции источников и возвращает формализованное описание каждой партиции.

    Функция не читает содержимое parquet-файлов. Она опирается только на структуру директорий и тем самым
    остаётся дешёвой даже при очень большом числе партиций. Возвращаемый результат уже содержит все поля,
    необходимые для последующего принятия решения `new / skip / fail`.

    Параметры:
        sources_root:
            Путь к директории `sources`.
        source_registry:
            Необязательный реестр источников. По умолчанию используется глобальный `SOURCE_REGISTRY`.

    Возвращаемое значение:
        Отсортированный список словарей. Каждый элемент содержит `source_name`, `source_id`, `partition_key`,
        `partition_value`, `partition_path`, `target_table` и `update_frequency`.

    Исключения:
        FileNotFoundError: если директория конкретного источника отсутствует.
        ValueError: если имя партиции не соответствует ожидаемому partition key.

    Пример входа:
        discover_source_partitions('/tmp/sources')

    Пример выхода:
        [
            {
                'source_name': 'credit_cards_info',
                'source_id': 2,
                'partition_key': 'report_dt',
                'partition_value': '2024-01-31',
                'partition_path': '/tmp/sources/credit_cards_info/report_dt='2024-01-31'',
                'target_table': 'client_monthly_attrs_scd1',
                'update_frequency': 'monthly'
            }
        ]
    """
    registry = source_registry or SOURCE_REGISTRY
    discovered_partitions: list[dict[str, Any]] = []
    sources_root_path = Path(sources_root)

    for source_name, source_meta in registry.items():
        source_dir = sources_root_path / source_name
        if not source_dir.exists():
            raise FileNotFoundError(f'Отсутствует директория источника: {source_dir}')

        for partition_dir in sorted(path for path in source_dir.iterdir() if path.is_dir() and not path.name.startswith('.')):
            partition_key, partition_value = parse_partition_directory_name(partition_dir.name)
            if partition_key != source_meta['partition_key']:
                raise ValueError(
                    f"Для источника {source_name} ожидался ключ {source_meta['partition_key']}, но найден {partition_key}."
                )
            discovered_partitions.append(
                {
                    'source_name': source_name,
                    'source_id': source_meta['source_id'],
                    'partition_key': partition_key,
                    'partition_value': partition_value,
                    'partition_path': str(partition_dir.resolve()),
                    'target_table': source_meta['target_table'],
                    'update_frequency': source_meta['update_frequency'],
                }
            )

    return sorted(
        discovered_partitions,
        key=lambda row: (row['source_id'], row['partition_value'], row['partition_path']),
    )

In [ ]:
discovered_example_partitions = discover_source_partitions(EXAMPLE_SOURCES_ROOT)
print(len(discovered_example_partitions))
print(discovered_example_partitions[0])

In [ ]:
def build_manifest_fingerprint(partition_path: str | Path) -> str:
    """
    Строит детерминированный fingerprint партиции по файловому manifest-описанию без чтения parquet-данных.

    В fingerprint включаются только файлы `*.parquet`, потому что `_SUCCESS` и другие технические файлы
    не должны влиять на решение о переобработке. Для каждого parquet-файла учитываются имя, размер и
    `mtime_ns`. Такой подход дешёв по I/O и позволяет отсеять уже обработанные партиции до запуска Spark-чтения.

    Параметры:
        partition_path:
            Путь к директории одной партиции источника.

    Возвращаемое значение:
        SHA-256 fingerprint в виде шестнадцатеричной строки.

    Исключения:
        FileNotFoundError: если в директории нет parquet-файлов.

    Пример входа:
        build_manifest_fingerprint('/tmp/sources/credit_cards_info/report_dt='2024-01-31'')

    Пример выхода:
        '5ff0a0f8f4f51ab3b6d8f0d6f53c8f7b6a3d6b0f7f2b4f3b3d2c1a0e9f8d7c6'
    """
    partition_dir = Path(partition_path)
    parquet_files = sorted(partition_dir.glob('*.parquet'))
    if not parquet_files:
        raise FileNotFoundError(f'В директории {partition_dir} не найдено parquet-файлов.')

    manifest_rows = []
    for parquet_file in parquet_files:
        stats = parquet_file.stat()
        manifest_rows.append(
            {
                'name': parquet_file.name,
                'size': stats.st_size,
                'mtime_ns': stats.st_mtime_ns,
            }
        )

    payload = json.dumps(manifest_rows, ensure_ascii=False, sort_keys=True).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

In [ ]:
EXAMPLE_MANIFEST_DIR = EXAMPLE_RUNTIME_ROOT / 'manifest_example'
EXAMPLE_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
(EXAMPLE_MANIFEST_DIR / 'part-00000.parquet').write_bytes(b'example-parquet-a')
(EXAMPLE_MANIFEST_DIR / 'part-00001.parquet').write_bytes(b'example-parquet-b')
print(build_manifest_fingerprint(EXAMPLE_MANIFEST_DIR))

## 4. Справочники и SQL-шаблоны

In [ ]:
def infer_attribute_data_type(attribute_name: str) -> str:
    """
    Определяет бизнес-тип атрибута по его имени и принятому naming convention.

    Это не физическая схема исходного parquet-файла, а именно тип логического атрибута для справочника
    `dim_attributes`. Функция нужна для автоматического наполнения небольшого справочника без ручного
    перечисления типа для каждой колонки.

    Параметры:
        attribute_name:
            Имя атрибута из `SOURCE_REGISTRY`.

    Возвращаемое значение:
        Одно из строковых значений `DATE`, `DECIMAL`, `INT`, `STRING`.

    Пример входа:
        infer_attribute_data_type('client_income_amt')

    Пример выхода:
        'DECIMAL'
    """
    if attribute_name.endswith(('_dt', '_from', '_to')):
        return 'DATE'
    if attribute_name.endswith('_amt'):
        return 'DECIMAL'
    if attribute_name.endswith(('_nflag', '_nfalg', '_qty')):
        return 'INT'
    return 'STRING'

In [ ]:
print({name: infer_attribute_data_type(name) for name in ['client_income_amt', 'dep_acc_1st_open_dt', 'cc_ever_nflag']})

In [ ]:
def initialize_warehouse_tables(spark, warehouse_root: str | Path) -> dict[str, str]:
    """
    Создаёт все внешние parquet-таблицы хранилища по заранее зафиксированной физической схеме.

    Функция идемпотентна: если таблица уже существует, DDL не пересоздаёт её и не затрагивает имеющиеся данные.
    Таблицы фактов создаются как partitioned external tables, а маленькие таблицы журнала и метаданных — как
    обычные external tables. Все физические location-пути размещаются под единым `warehouse_root`.

    Параметры:
        spark:
            Активная `SparkSession`.
        warehouse_root:
            Корневой каталог path-based хранилища.

    Возвращаемое значение:
        Словарь `{table_name: table_location}` для всех таблиц хранилища.

    Пример входа:
        initialize_warehouse_tables(example_spark, '/tmp/warehouse_debug')

    Пример выхода:
        {
            'dim_sources': '/tmp/warehouse_debug/dim_sources',
            'client_monthly_attrs_scd1': '/tmp/warehouse_debug/client_monthly_attrs_scd1',
            ...
        }
    """
    warehouse_root_path = Path(warehouse_root).resolve()
    warehouse_root_path.mkdir(parents=True, exist_ok=True)
    created_locations: dict[str, str] = {}

    for table_name, table_spec in WAREHOUSE_TABLE_SPECS.items():
        table_location = warehouse_root_path / table_name
        all_columns = [
            f"{column_name} {column_type}"
            for column_name, column_type in table_spec['columns']
        ]
        if spark.catalog.tableExists(table_name):
            describe_rows = spark.sql(f'DESCRIBE TABLE EXTENDED {table_name}').collect()
            location_value = next(
                (
                    row.data_type
                    for row in describe_rows
                    if str(row.col_name).strip().lower() == 'location'
                ),
                None,
            )
            if location_value is not None:
                existing_location = Path(str(location_value).replace('file:', '')).resolve()
                if existing_location != table_location.resolve():
                    spark.sql(f'DROP TABLE {table_name}')
        ddl = [
            f"CREATE TABLE IF NOT EXISTS {table_name}",
            f"({', '.join(all_columns)})",
            'USING parquet',
        ]
        if table_spec['partitioned_by']:
            ddl.append(f"PARTITIONED BY ({', '.join(table_spec['partitioned_by'])})")
        ddl.append(f"LOCATION '{str(table_location).replace("'", "''")}'")
        spark.sql(' '.join(ddl))
        spark.sql(f'REFRESH TABLE {table_name}')
        created_locations[table_name] = str(table_location)

    return created_locations

In [ ]:
example_table_locations = initialize_warehouse_tables(example_spark, example_paths['active_warehouse_root'])
print(sorted(example_table_locations))

In [ ]:
def upsert_reference_dimensions(spark, load_timestamp: dt.datetime) -> dict[str, int]:
    """
    Пересобирает маленькие справочники `dim_sources` и `dim_attributes` на основании текущего реестра источников.

    Операция выполняется через `INSERT OVERWRITE`, потому что размеры справочников малы и их проще полностью
    пересчитать, чем поддерживать точечные изменения. Такой компромисс безопасен и не влияет на масштабируемость,
    поскольку heavy-таблицы фактов при этом не перечитываются и не переписываются.

    Параметры:
        spark:
            Активная `SparkSession`.
        load_timestamp:
            Единая временная метка текущего запуска, используемая в технических полях справочников.

    Возвращаемое значение:
        Словарь с числом строк, записанных в `dim_sources` и `dim_attributes`.

    Пример входа:
        upsert_reference_dimensions(example_spark, dt.datetime(2024, 1, 31, 12, 0, 0))

    Пример выхода:
        {'dim_sources_rows': 3, 'dim_attributes_rows': 24}
    """
    dim_sources_rows = []
    for source_name, source_meta in SOURCE_REGISTRY.items():
        dim_sources_rows.append(
            {
                'source_id': source_meta['source_id'],
                'source_name': source_name,
                'source_description': source_meta['source_description'],
                'update_frequency': source_meta['update_frequency'],
                'row_create_dtime': load_timestamp,
                'row_update_dtime': load_timestamp,
                'valid_from': load_timestamp,
                'valid_to': FUTURE_DTTM,
            }
        )

    dim_attributes_rows = []
    for source_name, source_meta in SOURCE_REGISTRY.items():
        for attribute_name in source_meta['columns']:
            dim_attributes_rows.append(
                {
                    'attribute_id': ATTRIBUTE_ID_BY_NAME[attribute_name],
                    'attribute_name': attribute_name,
                    'attribute_description': f'Атрибут {attribute_name} из источника {source_name}.',
                    'data_type': infer_attribute_data_type(attribute_name),
                    'source_id': source_meta['source_id'],
                    'update_frequency': source_meta['update_frequency'],
                    'row_create_dtime': load_timestamp,
                    'row_update_dtime': load_timestamp,
                }
            )

    spark.createDataFrame(dim_sources_rows).createOrReplaceTempView('stage_dim_sources')
    spark.createDataFrame(dim_attributes_rows).createOrReplaceTempView('stage_dim_attributes')
    spark.sql('INSERT OVERWRITE TABLE dim_sources BY NAME SELECT * FROM stage_dim_sources')
    spark.sql('INSERT OVERWRITE TABLE dim_attributes BY NAME SELECT * FROM stage_dim_attributes')
    spark.sql('REFRESH TABLE dim_sources')
    spark.sql('REFRESH TABLE dim_attributes')

    return {
        'dim_sources_rows': len(dim_sources_rows),
        'dim_attributes_rows': len(dim_attributes_rows),
    }

In [ ]:
reference_summary = upsert_reference_dimensions(example_spark, dt.datetime(2024, 1, 31, 12, 0, 0))
print(reference_summary)

In [ ]:
def load_partition_state_map(spark) -> dict[tuple[str, str, str], dict[str, Any]]:
    """
    Загружает текущее состояние уже обработанных партиций из `tech_source_partitions` в компактный Python-словарь.

    Таблица технических состояний мала по объёму относительно факт-таблиц. Поэтому её допустимо целиком
    материализовать в память одного драйвера и далее принимать решение `new / skip / fail` без дополнительных
    запросов к Spark для каждой отдельной партиции.

    Параметры:
        spark:
            Активная `SparkSession`.

    Возвращаемое значение:
        Словарь с ключом `(source_name, partition_key, partition_value)` и словарём состояния в значении.

    Пример входа:
        load_partition_state_map(example_spark)

    Пример выхода:
        {
            ('credit_cards_info', 'report_dt', '2024-01-31'): {
                'manifest_fingerprint': '...',
                'last_processed_status': 'loaded',
                ...
            }
        }
    """
    rows = spark.sql('SELECT * FROM tech_source_partitions').collect()
    state_map: dict[tuple[str, str, str], dict[str, Any]] = {}
    for row in rows:
        row_dict = row.asDict(recursive=True)
        state_map[(row_dict['source_name'], row_dict['partition_key'], row_dict['partition_value'])] = row_dict
    return state_map

In [ ]:
print(load_partition_state_map(example_spark))

In [ ]:
def determine_partition_action(
    existing_state: dict[str, Any] | None,
    current_fingerprint: str,
) -> str:
    """
    Определяет действие для партиции: загрузить её, пропустить или аварийно завершить запуск.

    Правила intentionally жёсткие:
    - если партиция ранее не встречалась, возвращается `new`;
    - если fingerprint совпадает, возвращается `skip`;
    - если партиция уже есть, но fingerprint изменился, возвращается `fail`.

    Такой контракт предотвращает тихую порчу истории и позволяет быстро обнаружить изменение уже однажды
    обработанного upstream-среза.

    Параметры:
        existing_state:
            Словарь состояния из `tech_source_partitions` или `None`, если партиция ещё не встречалась.
        current_fingerprint:
            Новый manifest fingerprint, вычисленный по текущему каталогу партиции.

    Возвращаемое значение:
        Строка `new`, `skip` или `fail`.

    Пример входа:
        determine_partition_action({'manifest_fingerprint': 'abc'}, 'abc')

    Пример выхода:
        'skip'
    """
    if existing_state is None:
        return 'new'
    if existing_state['manifest_fingerprint'] == current_fingerprint:
        return 'skip'
    return 'fail'

In [ ]:
print(determine_partition_action(None, 'abc'))
print(determine_partition_action({'manifest_fingerprint': 'abc'}, 'abc'))
print(determine_partition_action({'manifest_fingerprint': 'abc'}, 'xyz'))

In [ ]:
def build_stage_sql(raw_view_name: str, source_name: str, partition_value: str) -> str:
    """
    Генерирует Spark SQL для преобразования wide-таблицы источника в целевую EAV-структуру.

    В monthly-источниках SQL формирует колонки для `client_monthly_attrs_scd1`, а в daily-источнике — для
    `client_daily_attrs_scd2`. Преобразование выполняется через `STACK` и join со справочником `dim_attributes`,
    что делает соответствие `attribute_name -> attribute_id` явно управляемым и воспроизводимым.

    Параметры:
        raw_view_name:
            Имя временного view с сырыми данными одной партиции.
        source_name:
            Имя источника из `SOURCE_REGISTRY`.
        partition_value:
            Значение партиции, которое будет записано в `report_dt` или `row_actual_to`.

    Возвращаемое значение:
        Строка Spark SQL, готовая для `CREATE OR REPLACE TEMP VIEW ... AS`.

    Пример входа:
        build_stage_sql('raw_credit_cards_info', 'credit_cards_info', '2024-01-31')

    Пример выхода:
        SQL-строка, содержащая `LATERAL VIEW STACK(...)` и join к `dim_attributes`.
    """
    source_meta = SOURCE_REGISTRY[source_name]
    source_id = source_meta['source_id']
    escaped_partition_value = partition_value.replace("'", "''")
    stack_arguments = ',\n        '.join(
        f"'{column_name}', CAST({column_name} AS STRING)"
        for column_name in source_meta['columns']
    )

    if source_meta['update_frequency'] == 'monthly':
        return f"""
        SELECT
            staged_rows.client_id AS client_id,
            attrs.attribute_id AS attribute_id,
            staged_rows.attribute_value AS attribute_value,
            {source_id} AS source_id,
            staged_rows.row_update_dtime AS row_update_dtime,
            staged_rows.row_loading_id AS row_loading_id,
            staged_rows.row_hash_val AS row_hash_val,
            '{escaped_partition_value}' AS report_dt
        FROM (
            SELECT
                CAST(src.id AS STRING) AS client_id,
                CAST(staged.attribute_value AS STRING) AS attribute_value,
                CAST(src.row_update_dtime AS TIMESTAMP) AS row_update_dtime,
                CAST(src.loading_id AS BIGINT) AS row_loading_id,
                CAST(src.row_hash_val AS STRING) AS row_hash_val,
                staged.attribute_name AS attribute_name
            FROM {raw_view_name} AS src
            LATERAL VIEW STACK(
                {len(source_meta['columns'])},
                {stack_arguments}
            ) staged AS attribute_name, attribute_value
        ) AS staged_rows
        INNER JOIN dim_attributes AS attrs
            ON attrs.source_id = {source_id}
           AND attrs.attribute_name = staged_rows.attribute_name
        """.strip()

    return f"""
    SELECT
        staged_rows.client_id AS client_id,
        attrs.attribute_id AS attribute_id,
        staged_rows.attribute_value AS attribute_value,
        staged_rows.row_actual_from AS row_actual_from,
        {source_id} AS source_id,
        staged_rows.row_update_dtime AS row_update_dtime,
        staged_rows.row_loading_id AS row_loading_id,
        staged_rows.row_hash_val AS row_hash_val,
        '{escaped_partition_value}' AS row_actual_to
    FROM (
        SELECT
            CAST(src.id AS STRING) AS client_id,
            CAST(staged.attribute_value AS STRING) AS attribute_value,
            CAST(src.row_actual_from AS STRING) AS row_actual_from,
            CAST(src.row_update_dtime AS TIMESTAMP) AS row_update_dtime,
            CAST(src.loading_id AS BIGINT) AS row_loading_id,
            CAST(src.row_hash_val AS STRING) AS row_hash_val,
            staged.attribute_name AS attribute_name
        FROM {raw_view_name} AS src
        LATERAL VIEW STACK(
            {len(source_meta['columns'])},
            {stack_arguments}
        ) staged AS attribute_name, attribute_value
    ) AS staged_rows
    INNER JOIN dim_attributes AS attrs
        ON attrs.source_id = {source_id}
       AND attrs.attribute_name = staged_rows.attribute_name
    """.strip()

In [ ]:
print('\n'.join(build_stage_sql('raw_credit_cards_info', 'credit_cards_info', '2024-01-31').splitlines()[:10]))


## 5. Запись фактов и технических журналов

In [ ]:
def append_partition_to_fact_table(spark, partition_record: dict[str, Any], load_id: int) -> int:
    """
    Читает одну новую партицию источника, строит staging-view и append-only записывает её в целевую факт-таблицу.

    Функция применяется только к партициям, которые уже классифицированы как `new`. Она не проверяет повторность
    сама по себе: это обязанность оркестратора. Возвращаемое число строк используется для технического лога и
    не требует дополнительного чтения целевой таблицы.

    Параметры:
        spark:
            Активная `SparkSession`.
        partition_record:
            Словарь, полученный из `discover_source_partitions`.
        load_id:
            Идентификатор текущего загрузочного события.

    Возвращаемое значение:
        Число строк, записанных в целевую факт-таблицу.

    Пример входа:
        append_partition_to_fact_table(example_spark, partition_record, load_id=101)

    Пример выхода:
        22
    """
    source_name = partition_record['source_name']
    raw_view_name = re.sub(r'[^A-Za-z0-9_]+', '_', f"raw_{source_name}_{partition_record['partition_key']}_{partition_record['partition_value']}_{load_id}")
    stage_view_name = re.sub(r'[^A-Za-z0-9_]+', '_', f"stage_{source_name}_{partition_record['partition_key']}_{partition_record['partition_value']}_{load_id}")

    spark.read.parquet(partition_record['partition_path']).createOrReplaceTempView(raw_view_name)
    stage_sql = build_stage_sql(raw_view_name, source_name, partition_record['partition_value'])
    spark.sql(f'CREATE OR REPLACE TEMP VIEW {stage_view_name} AS {stage_sql}')

    rows_loaded = int(
        spark.sql(f'SELECT COUNT(*) AS rows_loaded FROM {stage_view_name}').collect()[0]['rows_loaded']
    )
    spark.sql(f"INSERT INTO {partition_record['target_table']} BY NAME SELECT * FROM {stage_view_name}")

    try:
        spark.catalog.dropTempView(stage_view_name)
    except Exception:
        pass
    try:
        spark.catalog.dropTempView(raw_view_name)
    except Exception:
        pass

    return rows_loaded

In [ ]:
monthly_partition_dir = EXAMPLE_SOURCES_ROOT / 'credit_cards_info' / "report_dt='2024-01-31'"
monthly_partition_dir.mkdir(parents=True, exist_ok=True)
monthly_example_rows = [
    {
        'id': 'C001',
        'client_income_amt': 100000.0,
        'oi_total_amt': 150000.0,
        'act_pl_os_rub_amt': 75000.0,
        'payroll_client_nflag': 1,
        'inf_payroll_rub_amt': 50000.0,
        'legal_entity_amt': 25000.0,
        'inc_avg_risk_rub_amt': 1500.0,
        'otf_loan_rub_amt': 5000.0,
        'otf_fee_rub_amt': 250.0,
        'inf_transfer_rub_amt': 3000.0,
        'cc_ever_nflag': 1,
        'row_update_dtime': dt.datetime(2024, 1, 31, 10, 0, 0),
        'loading_id': 101,
        'row_hash_val': 'hash-credit-20240131-C001',
    }
]
example_spark.createDataFrame(monthly_example_rows).write.mode('overwrite').parquet(str(monthly_partition_dir))
monthly_partition_record = next(
    item
    for item in discover_source_partitions(EXAMPLE_SOURCES_ROOT)
    if item['source_name'] == 'credit_cards_info' and item['partition_value'] == '2024-01-31'
)
print(append_partition_to_fact_table(example_spark, monthly_partition_record, load_id=101))

In [ ]:
def append_load_log_entry(spark, load_log_row: dict[str, Any]) -> None:
    """
    Добавляет одну запись в `load_log` через временный staging-view и `INSERT INTO`.

    Журнал загрузок является append-only таблицей. Каждое решение оркестратора (`loaded`, `skipped`, `failed`)
    обязано попадать в лог отдельной строкой, чтобы поведение процесса можно было анализировать постфактум.

    Параметры:
        spark:
            Активная `SparkSession`.
        load_log_row:
            Словарь со всеми колонками `load_log`.

    Возвращаемое значение:
        `None`.

    Пример входа:
        append_load_log_entry(example_spark, {...})

    Пример выхода:
        Запись появляется в таблице `load_log`; функция ничего не возвращает.
    """
    load_log_schema = ', '.join(
        f"{column_name} {column_type}"
        for column_name, column_type in WAREHOUSE_TABLE_SPECS['load_log']['columns']
    )
    spark.createDataFrame([load_log_row], schema=load_log_schema).createOrReplaceTempView('stage_single_load_log_row')
    spark.sql('INSERT INTO load_log BY NAME SELECT * FROM stage_single_load_log_row')
    spark.sql('REFRESH TABLE load_log')

In [ ]:
append_load_log_entry(
    example_spark,
    {
        'load_id': 101,
        'source_id': 2,
        'source_name': 'credit_cards_info',
        'source_partition_key': 'report_dt',
        'source_partition_value': '2024-01-31',
        'target_table': 'client_monthly_attrs_scd1',
        'load_start_dtime': dt.datetime(2024, 1, 31, 10, 0, 0),
        'load_end_dtime': dt.datetime(2024, 1, 31, 10, 0, 1),
        'load_status': 'loaded',
        'rows_loaded': 11,
        'error_message': None,
    },
)
example_spark.sql('SELECT load_id, load_status, rows_loaded FROM load_log ORDER BY load_id DESC LIMIT 1').show(truncate=False)

In [ ]:
def merge_partition_state_entry(spark, state_row: dict[str, Any]) -> None:
    """
    Выполняет upsert одной записи в таблицу `tech_source_partitions` через SQL-overwrite малого технического набора.

    Таблица технических состояний намеренно остаётся маленькой. Поэтому для неё допустим алгоритм
    `snapshot + union + row_number + insert overwrite`, который был бы неприемлем для тяжёлых факт-таблиц,
    но отлично подходит для служебного контура идемпотентности.

    Параметры:
        spark:
            Активная `SparkSession`.
        state_row:
            Словарь со всеми колонками `tech_source_partitions`.

    Возвращаемое значение:
        `None`.

    Пример входа:
        merge_partition_state_entry(example_spark, {...})

    Пример выхода:
        В `tech_source_partitions` остаётся одна актуальная запись на комбинацию
        `(source_name, partition_key, partition_value)`.
    """
    partition_state_schema = ', '.join(
        f"{column_name} {column_type}"
        for column_name, column_type in WAREHOUSE_TABLE_SPECS['tech_source_partitions']['columns']
    )
    existing_partition_state_rows = [
        row.asDict(recursive=True)
        for row in spark.sql('SELECT * FROM tech_source_partitions').collect()
    ]
    spark.createDataFrame(
        existing_partition_state_rows,
        schema=partition_state_schema,
    ).createOrReplaceTempView('tech_source_partitions_snapshot')
    spark.createDataFrame([state_row], schema=partition_state_schema).createOrReplaceTempView('stage_single_partition_state_row')
    spark.sql(
        """
        INSERT OVERWRITE TABLE tech_source_partitions BY NAME
        SELECT
            source_id,
            source_name,
            target_table,
            partition_key,
            partition_value,
            partition_path,
            manifest_fingerprint,
            last_processed_status,
            first_load_id,
            last_load_id,
            row_create_dtime,
            row_update_dtime
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY source_name, partition_key, partition_value
                    ORDER BY row_update_dtime DESC, last_load_id DESC
                ) AS row_rank
            FROM (
                SELECT
                    source_id,
                    source_name,
                    target_table,
                    partition_key,
                    partition_value,
                    partition_path,
                    manifest_fingerprint,
                    last_processed_status,
                    first_load_id,
                    last_load_id,
                    row_create_dtime,
                    row_update_dtime
                FROM tech_source_partitions_snapshot
                UNION ALL
                SELECT
                    source_id,
                    source_name,
                    target_table,
                    partition_key,
                    partition_value,
                    partition_path,
                    manifest_fingerprint,
                    last_processed_status,
                    first_load_id,
                    last_load_id,
                    row_create_dtime,
                    row_update_dtime
                FROM stage_single_partition_state_row
            ) AS unioned_state_rows
        ) AS ranked_state_rows
        WHERE row_rank = 1
        """
    )
    spark.sql('REFRESH TABLE tech_source_partitions')

In [ ]:
merge_partition_state_entry(
    example_spark,
    {
        'source_id': 2,
        'source_name': 'credit_cards_info',
        'target_table': 'client_monthly_attrs_scd1',
        'partition_key': 'report_dt',
        'partition_value': '2024-01-31',
        'partition_path': str(monthly_partition_dir),
        'manifest_fingerprint': build_manifest_fingerprint(monthly_partition_dir),
        'last_processed_status': 'loaded',
        'first_load_id': 101,
        'last_load_id': 101,
        'row_create_dtime': dt.datetime(2024, 1, 31, 10, 0, 0),
        'row_update_dtime': dt.datetime(2024, 1, 31, 10, 0, 1),
    },
)
print(load_partition_state_map(example_spark))

## 6. Безопасная очистка debug-артефактов и единый orchestration-run

In [ ]:
def clear_debug_artifacts(target_root: str | Path, approved_debug_root: str | Path) -> str:
    """
    Полностью удаляет каталог debug-артефактов, если и только если путь прошёл строгую проверку безопасности.

    Функция защищает от опасных сценариев вида «случайно передали production-root вместо debug-root».
    Для удаления путь должен:
    - совпадать с явно разрешённым `approved_debug_root`;
    - содержать в имени маркер `debug`;
    - не быть системным корнем файловой системы.

    Параметры:
        target_root:
            Путь, который нужно очистить.
        approved_debug_root:
            Путь, который заранее признан безопасным debug-root каталога.

    Возвращаемое значение:
        Нормализованный путь к удалённому каталогу.

    Исключения:
        ValueError: если путь не прошёл проверку безопасности.

    Пример входа:
        clear_debug_artifacts('/tmp/warehouse_debug', '/tmp/warehouse_debug')

    Пример выхода:
        '/tmp/warehouse_debug'
    """
    target_path = Path(target_root).resolve()
    approved_path = Path(approved_debug_root).resolve()
    if target_path != approved_path:
        raise ValueError('Разрешено удалять только заранее подтверждённый debug-root каталог.')
    if 'debug' not in target_path.name.lower():
        raise ValueError('Для удаления каталог должен явно содержать маркер debug в имени.')
    if str(target_path) in {'/', str(Path.home().resolve())}:
        raise ValueError('Удаление системных корней запрещено.')
    if target_path.exists():
        shutil.rmtree(target_path)
    return str(target_path)

In [ ]:
debug_cleanup_root = EXAMPLE_RUNTIME_ROOT / 'safe_debug_directory'
debug_cleanup_root.mkdir(parents=True, exist_ok=True)
(debug_cleanup_root / 'temporary.txt').write_text('debug payload', encoding='utf-8')
print(clear_debug_artifacts(debug_cleanup_root, debug_cleanup_root))
print(debug_cleanup_root.exists())

In [ ]:
def run_warehouse_update(
    spark,
    sources_root: str | Path | None = None,
    warehouse_root: str | Path | None = None,
    run_mode: str = 'production',
    debug_root: str | Path | None = None,
    cleanup_existing_debug_root: bool = False,
) -> dict[str, Any]:
    """
    Выполняет полный инкрементальный цикл обновления path-based хранилища данных.

    Алгоритм запуска:
    1. Разрешает пути и режим работы.
    2. При необходимости безопасно очищает debug-root.
    3. Инициализирует внешние таблицы хранилища.
    4. Полностью пересобирает маленькие справочники `dim_sources` и `dim_attributes`.
    5. Загружает текущее состояние `tech_source_partitions`.
    6. Для каждой найденной партиции вычисляет manifest fingerprint и принимает решение:
       - `new`: читает parquet и append-only пишет данные в факт-таблицу;
       - `skip`: не читает parquet повторно и пишет только audit-записи;
       - `fail`: фиксирует ошибку в журнале и аварийно завершает запуск.

    Параметры:
        spark:
            Активная `SparkSession`.
        sources_root:
            Необязательный путь к директории `sources`.
        warehouse_root:
            Необязательный production-root каталога хранилища.
        run_mode:
            `production` или `debug`.
        debug_root:
            Необязательный отдельный debug-root каталог.
        cleanup_existing_debug_root:
            Флаг, разрешающий полную очистку только debug-root каталога перед запуском.

    Возвращаемое значение:
        Сводный словарь с числом загруженных, пропущенных и неуспешных партиций, а также с location-ами таблиц.

    Пример входа:
        run_warehouse_update(example_spark, '/tmp/sources', '/tmp/warehouse_prod', 'debug', '/tmp/warehouse_debug', True)

    Пример выхода:
        {
            'run_mode': 'debug',
            'loaded_partitions': 3,
            'skipped_partitions': 0,
            'failed_partitions': 0,
            'processed_partitions': [...],
            'table_locations': {...}
        }
    """
    runtime_paths = resolve_runtime_paths(
        sources_root=sources_root,
        warehouse_root=warehouse_root,
        run_mode=run_mode,
        debug_root=debug_root,
    )
    active_warehouse_root = Path(runtime_paths['active_warehouse_root'])
    if runtime_paths['run_mode'] == 'debug' and cleanup_existing_debug_root:
        clear_debug_artifacts(active_warehouse_root, runtime_paths['debug_warehouse_root'])

    table_locations = initialize_warehouse_tables(spark, active_warehouse_root)
    run_timestamp = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
    upsert_reference_dimensions(spark, run_timestamp)

    partition_state_map = load_partition_state_map(spark)
    discovered_partitions = discover_source_partitions(runtime_paths['sources_root'])
    current_load_id = int(spark.sql('SELECT COALESCE(MAX(load_id), 0) AS max_load_id FROM load_log').collect()[0]['max_load_id'])

    processed_partitions = []
    loaded_partitions = 0
    skipped_partitions = 0
    failed_partitions = 0

    for partition_record in discovered_partitions:
        current_load_id += 1
        load_start = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
        partition_key = (
            partition_record['source_name'],
            partition_record['partition_key'],
            partition_record['partition_value'],
        )
        current_fingerprint = build_manifest_fingerprint(partition_record['partition_path'])
        existing_state = partition_state_map.get(partition_key)
        action = determine_partition_action(existing_state, current_fingerprint)

        if action == 'skip':
            skipped_partitions += 1
            append_load_log_entry(
                spark,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                    'load_status': 'skipped',
                    'rows_loaded': 0,
                    'error_message': None,
                },
            )
            state_row = dict(existing_state)
            state_row['last_processed_status'] = 'skipped'
            state_row['last_load_id'] = current_load_id
            state_row['row_update_dtime'] = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            merge_partition_state_entry(spark, state_row)
            partition_state_map[partition_key] = state_row
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'skipped',
                    'rows_loaded': 0,
                }
            )
            continue

        if action == 'fail':
            failed_partitions += 1
            error_message = (
                'Обнаружено изменение уже обработанной партиции: '
                f"{partition_record['source_name']} / {partition_record['partition_key']}={partition_record['partition_value']}"
            )
            append_load_log_entry(
                spark,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                    'load_status': 'failed',
                    'rows_loaded': 0,
                    'error_message': error_message,
                },
            )
            raise RuntimeError(error_message)

        try:
            rows_loaded = append_partition_to_fact_table(spark, partition_record, current_load_id)
            loaded_partitions += 1
            load_end = dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0)
            append_load_log_entry(
                spark,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': load_end,
                    'load_status': 'loaded',
                    'rows_loaded': rows_loaded,
                    'error_message': None,
                },
            )
            state_row = {
                'source_id': partition_record['source_id'],
                'source_name': partition_record['source_name'],
                'target_table': partition_record['target_table'],
                'partition_key': partition_record['partition_key'],
                'partition_value': partition_record['partition_value'],
                'partition_path': partition_record['partition_path'],
                'manifest_fingerprint': current_fingerprint,
                'last_processed_status': 'loaded',
                'first_load_id': existing_state['first_load_id'] if existing_state else current_load_id,
                'last_load_id': current_load_id,
                'row_create_dtime': existing_state['row_create_dtime'] if existing_state else load_start,
                'row_update_dtime': load_end,
            }
            merge_partition_state_entry(spark, state_row)
            partition_state_map[partition_key] = state_row
            processed_partitions.append(
                {
                    'source_name': partition_record['source_name'],
                    'partition_value': partition_record['partition_value'],
                    'action': 'loaded',
                    'rows_loaded': rows_loaded,
                }
            )
        except Exception as exc:
            failed_partitions += 1
            append_load_log_entry(
                spark,
                {
                    'load_id': current_load_id,
                    'source_id': partition_record['source_id'],
                    'source_name': partition_record['source_name'],
                    'source_partition_key': partition_record['partition_key'],
                    'source_partition_value': partition_record['partition_value'],
                    'target_table': partition_record['target_table'],
                    'load_start_dtime': load_start,
                    'load_end_dtime': dt.datetime.now(dt.UTC).replace(tzinfo=None, microsecond=0),
                    'load_status': 'failed',
                    'rows_loaded': 0,
                    'error_message': str(exc),
                },
            )
            raise

    return {
        'run_mode': runtime_paths['run_mode'],
        'sources_root': runtime_paths['sources_root'],
        'active_warehouse_root': str(active_warehouse_root),
        'loaded_partitions': loaded_partitions,
        'skipped_partitions': skipped_partitions,
        'failed_partitions': failed_partitions,
        'processed_partitions': processed_partitions,
        'table_locations': table_locations,
    }

In [ ]:
EXAMPLE_FULL_ROOT = EXAMPLE_RUNTIME_ROOT / 'full_pipeline'
EXAMPLE_FULL_SOURCES = EXAMPLE_FULL_ROOT / 'sources'
EXAMPLE_FULL_DEBUG = EXAMPLE_FULL_ROOT / 'warehouse_debug'
for source_name in SOURCE_REGISTRY:
    (EXAMPLE_FULL_SOURCES / source_name).mkdir(parents=True, exist_ok=True)

example_credit_rows = [
    {
        'id': 'C100',
        'client_income_amt': 120000.0,
        'oi_total_amt': 190000.0,
        'act_pl_os_rub_amt': 80000.0,
        'payroll_client_nflag': 1,
        'inf_payroll_rub_amt': 62000.0,
        'legal_entity_amt': 24000.0,
        'inc_avg_risk_rub_amt': 900.0,
        'otf_loan_rub_amt': 4000.0,
        'otf_fee_rub_amt': 150.0,
        'inf_transfer_rub_amt': 1000.0,
        'cc_ever_nflag': 1,
        'row_update_dtime': dt.datetime(2024, 3, 31, 9, 0, 0),
        'loading_id': 201,
        'row_hash_val': 'credit-20240331-C100',
    }
]
example_debit_rows = [
    {
        'id': 'C100',
        'onl_bank_active_1m_nfalg': 1,
        'auto_pay_active_qty': 2,
        'cl_income_1m_amt': 70000.0,
        'dep_acc_1st_open_dt': '2021-06-01',
        'wdr_cash_6m_amt': 12000.0,
        'cash_op_6m_amt': 14000.0,
        'cash_3m_qty': 4,
        'lst_balance_amt': 30000.0,
        'card_active_1m_nflag': 1,
        'row_update_dtime': dt.datetime(2024, 3, 31, 9, 5, 0),
        'loading_id': 202,
        'row_hash_val': 'debit-20240331-C100',
    }
]
example_daily_rows = [
    {
        'id': 'C100',
        'srv_mb_nflag': 1,
        'cc_stoplist': 0,
        'lne_tot_debt_int_ovrd_rub_amt': 0.0,
        'lne_tot_debt_ovrd_rub_amt': 100.0,
        'row_actual_from': '2024-03-01',
        'row_update_dtime': dt.datetime(2024, 3, 31, 9, 10, 0),
        'loading_id': 203,
        'row_hash_val': 'daily-20240331-C100',
    }
]

example_spark.createDataFrame(example_credit_rows).write.mode('overwrite').parquet(
    str(EXAMPLE_FULL_SOURCES / 'credit_cards_info' / "report_dt='2024-03-31'")
)
example_spark.createDataFrame(example_debit_rows).write.mode('overwrite').parquet(
    str(EXAMPLE_FULL_SOURCES / 'deb_cards_info' / "report_dt='2024-03-31'")
)
example_spark.createDataFrame(example_daily_rows).write.mode('overwrite').parquet(
    str(EXAMPLE_FULL_SOURCES / 'client_cards_daily' / "row_actual_to='2024-03-31'")
)

example_run_summary = run_warehouse_update(
    example_spark,
    sources_root=EXAMPLE_FULL_SOURCES,
    warehouse_root=EXAMPLE_FULL_ROOT / 'warehouse_prod',
    run_mode='debug',
    debug_root=EXAMPLE_FULL_DEBUG,
    cleanup_existing_debug_root=True,
)
print(json.dumps(example_run_summary, ensure_ascii=False, indent=2))

## 7. Финальный пример запуска

По умолчанию notebook запускает безопасный `debug`-прогон и пишет артефакты в отдельный debug-root каталог.
При необходимости пути можно переопределить переменными окружения:
- `DBSCORING_SOURCES_ROOT`
- `DBSCORING_WAREHOUSE_ROOT`
- `DBSCORING_DEBUG_ROOT`
- `DBSCORING_RUN_MODE`
- `DBSCORING_CLEAN_DEBUG`
- `DBSCORING_SKIP_FINAL_RUN`

In [ ]:
if os.environ.get('DBSCORING_SKIP_FINAL_RUN', '0') == '1':
    print('Финальный прогон пропущен по флагу DBSCORING_SKIP_FINAL_RUN=1')
else:
    final_spark = example_spark
    final_summary = run_warehouse_update(
        final_spark,
        sources_root=os.environ.get('DBSCORING_SOURCES_ROOT'),
        warehouse_root=os.environ.get('DBSCORING_WAREHOUSE_ROOT'),
        run_mode=os.environ.get('DBSCORING_RUN_MODE', 'debug'),
        debug_root=os.environ.get('DBSCORING_DEBUG_ROOT'),
        cleanup_existing_debug_root=os.environ.get('DBSCORING_CLEAN_DEBUG', '1') == '1',
    )
    print(json.dumps(final_summary, ensure_ascii=False, indent=2))